Split TCRs into training, validation and test, in a way such that there is no overlap in terms of either alpha or beta chain among training, validation and test for positive, and no overlap for negative either. 

In [1]:
import numpy as np
import pandas as pd

import random

from collections import defaultdict
from collections import Counter

In [2]:
rseed = 1000
np_seed = 1216

random.seed(rseed)
np.random.seed(np_seed)

In [3]:
folder_path = "../data_gene_expression_signature/Zheng_2021_processed/"
file_pos = "TCRs_pos_1.csv"
file_neg = "TCRs_neg_1.csv"

df_pos = pd.read_csv(folder_path+file_pos, header=0)
df_neg = pd.read_csv(folder_path+file_neg, header=0)

print(df_pos.shape)
print(df_neg.shape)

print(df_pos[:2])
print(df_neg[:2])

(6390, 4)
(15749, 4)
                                               tcr  freq cell cancer_reactive
0          TRAV9-2,TRBV7-9,CALLDDKIIF,CASSLGAGEQFF   203  cd8             yes
1  TRAV13-1,TRBV5-8,CAASISGGYNKLIF,CASYPGTSGRGAQYF   137  cd8             yes
                                            tcr  freq cell cancer_reactive
0  TRAV12-1,TRBV29-1,CVVNQAGTALIF,CSVEGGFAYEQYF   139  cd8              no
1  TRAV19,TRBV11-3,CALVETGANNLFF,CASSLPGMNTEAFF   130  cd8              no


In [4]:
pos_tcrs = df_pos.tcr.tolist()
neg_tcrs = df_neg.tcr.tolist()

pos_ws = df_pos.freq.tolist()
neg_ws = df_neg.freq.tolist()

unique_pos_cdr3 = list(set([(x.split(",")[2], x.split(",")[3]) for x in pos_tcrs]))
unique_neg_cdr3 = list(set([(x.split(",")[2], x.split(",")[3]) for x in neg_tcrs]))

print("number of unique pos paired cdr3 is: "+str(len(unique_pos_cdr3)))
print("number of unique neg paired cdr3 is: "+str(len(unique_neg_cdr3)))

number of unique pos paired cdr3 is: 6357
number of unique neg paired cdr3 is: 15685


In [5]:
unique_pos_cdr3.sort()
unique_neg_cdr3.sort()

random.shuffle(unique_pos_cdr3)
random.shuffle(unique_neg_cdr3)

n_pos = len(unique_pos_cdr3)
n_neg = len(unique_neg_cdr3)

print(n_pos)
print(n_neg)

6357
15685


In [6]:
# set a starting proportion on the unique (alpha cdr3, beta cdr3) level
# to build test, validation and training datasets from
n_pos_test = round(0.12 * n_pos)
n_neg_test = round(0.12 * n_neg)

n_pos_valid = round(0.14 * n_pos)
n_neg_valid = round(0.14 * n_neg)

In [7]:
# select the (alpha cdr3, beta cdr3) combinations based on which to decide the test dataset
initial_unique_pos_cdr3_test = unique_pos_cdr3[:n_pos_test]
initial_unique_neg_cdr3_test = unique_neg_cdr3[:n_neg_test]

initial_unique_pos_alpha_cdr3_test_set = set([ab_cdr3[0] for ab_cdr3 in initial_unique_pos_cdr3_test])
initial_unique_neg_alpha_cdr3_test_set = set([ab_cdr3[0] for ab_cdr3 in initial_unique_neg_cdr3_test])

initial_unique_pos_beta_cdr3_test_set = set([ab_cdr3[1] for ab_cdr3 in initial_unique_pos_cdr3_test])
initial_unique_neg_beta_cdr3_test_set = set([ab_cdr3[1] for ab_cdr3 in initial_unique_neg_cdr3_test])

# print(len(initial_unique_pos_alpha_cdr3_test_set))
# print(len(initial_unique_neg_alpha_cdr3_test_set))

# print(len(initial_unique_pos_beta_cdr3_test_set))
# print(len(initial_unique_neg_beta_cdr3_test_set))

In [8]:
unique_pos_cdr3_test_list = [ab_cdr3 for ab_cdr3 in unique_pos_cdr3 if
                            ((ab_cdr3[0] in initial_unique_pos_alpha_cdr3_test_set) or
                             (ab_cdr3[1] in initial_unique_pos_beta_cdr3_test_set))]

unique_neg_cdr3_test_list = [ab_cdr3 for ab_cdr3 in unique_neg_cdr3 if
                            ((ab_cdr3[0] in initial_unique_neg_alpha_cdr3_test_set) or
                             (ab_cdr3[1] in initial_unique_neg_beta_cdr3_test_set))]

unique_pos_cdr3_test_set = set(unique_pos_cdr3_test_list)
unique_neg_cdr3_test_set = set(unique_neg_cdr3_test_list)

# print(len(unique_pos_cdr3_test_set))
# print(len(unique_neg_cdr3_test_set))

# list the alpha and beta chains from chosen test pairs, in order to exclude them 
# in validation and training

pos_alpha_to_exclude = set([x[0] for x in unique_pos_cdr3_test_list])
neg_alpha_to_exclude = set([x[0] for x in unique_neg_cdr3_test_list])

pos_beta_to_exclude = set([x[1] for x in unique_pos_cdr3_test_list])
neg_beta_to_exclude = set([x[1] for x in unique_neg_cdr3_test_list])

In [9]:
# select the (alpha cdr3, beta cdr3) combinations based on which to decide the validation dataset
unique_pos_cdr3_left = list(set(unique_pos_cdr3) - unique_pos_cdr3_test_set)
unique_neg_cdr3_left = list(set(unique_neg_cdr3) - unique_neg_cdr3_test_set)

# print(len(unique_pos_cdr3_left))
# print(len(unique_neg_cdr3_left))

unique_pos_cdr3_left.sort()
unique_neg_cdr3_left.sort()

random.shuffle(unique_pos_cdr3_left)
random.shuffle(unique_neg_cdr3_left)

initial_unique_pos_cdr3_valid = unique_pos_cdr3_left[:n_pos_valid]
initial_unique_neg_cdr3_valid = unique_neg_cdr3_left[:n_neg_valid]

initial_unique_pos_alpha_cdr3_valid_set = set([ab_cdr3[0] for ab_cdr3 in initial_unique_pos_cdr3_valid])
initial_unique_neg_alpha_cdr3_valid_set = set([ab_cdr3[0] for ab_cdr3 in initial_unique_neg_cdr3_valid])

initial_unique_pos_beta_cdr3_valid_set = set([ab_cdr3[1] for ab_cdr3 in initial_unique_pos_cdr3_valid])
initial_unique_neg_beta_cdr3_valid_set = set([ab_cdr3[1] for ab_cdr3 in initial_unique_neg_cdr3_valid])

# print(len(initial_unique_pos_alpha_cdr3_valid_set))
# print(len(initial_unique_neg_alpha_cdr3_valid_set))

# print(len(initial_unique_pos_beta_cdr3_valid_set))
# print(len(initial_unique_neg_beta_cdr3_valid_set))

unique_pos_cdr3_valid_list = [ab_cdr3 for ab_cdr3 in unique_pos_cdr3_left if
                              (((ab_cdr3[0] in initial_unique_pos_alpha_cdr3_valid_set) or
                                (ab_cdr3[1] in initial_unique_pos_beta_cdr3_valid_set)) and 
                               ((ab_cdr3[0] not in pos_alpha_to_exclude) and 
                                (ab_cdr3[1] not in pos_beta_to_exclude)))]

unique_neg_cdr3_valid_list = [ab_cdr3 for ab_cdr3 in unique_neg_cdr3_left if
                              (((ab_cdr3[0] in initial_unique_neg_alpha_cdr3_valid_set) or
                                (ab_cdr3[1] in initial_unique_neg_beta_cdr3_valid_set)) and 
                               ((ab_cdr3[0] not in neg_alpha_to_exclude) and 
                                (ab_cdr3[1] not in neg_beta_to_exclude)))]

unique_pos_cdr3_valid_set = set(unique_pos_cdr3_valid_list)
unique_neg_cdr3_valid_set = set(unique_neg_cdr3_valid_list)
# print(len(unique_pos_cdr3_valid_set))
# print(len(unique_neg_cdr3_valid_set))

In [10]:
# list the alpha and beta chains from chosen test and validation pairs, in order to exclude them 
# from training

pos_alpha_to_exclude = set([x[0] for x in unique_pos_cdr3_test_list+unique_pos_cdr3_valid_list])
neg_alpha_to_exclude = set([x[0] for x in unique_neg_cdr3_test_list+unique_neg_cdr3_valid_list])

pos_beta_to_exclude = set([x[1] for x in unique_pos_cdr3_test_list+unique_pos_cdr3_valid_list])
neg_beta_to_exclude = set([x[1] for x in unique_neg_cdr3_test_list+unique_neg_cdr3_valid_list])

In [11]:
# other (alpha cdr3, beta cdr3) combinations left are used to decide the training dataset

unique_pos_cdr3_train_set = set([ab_cdr3 for ab_cdr3 in unique_pos_cdr3 if 
                                 ((ab_cdr3[0] not in pos_alpha_to_exclude) and 
                                  (ab_cdr3[1] not in pos_beta_to_exclude))])

unique_neg_cdr3_train_set = set([ab_cdr3 for ab_cdr3 in unique_neg_cdr3 if 
                                ((ab_cdr3[0] not in neg_alpha_to_exclude) and 
                                 (ab_cdr3[1] not in neg_beta_to_exclude))])

In [12]:
train_pos_f = [(x, f) for x, f in list(zip(pos_tcrs, pos_ws)) if
               (x.split(",")[2], x.split(",")[3]) in unique_pos_cdr3_train_set]
train_neg_f = [(x, f) for x, f in list(zip(neg_tcrs, neg_ws)) if
               (x.split(",")[2], x.split(",")[3]) in unique_neg_cdr3_train_set]

valid_pos_f = [(x, f) for x, f in list(zip(pos_tcrs, pos_ws)) if
               (x.split(",")[2], x.split(",")[3]) in unique_pos_cdr3_valid_set]
valid_neg_f = [(x, f) for x, f in list(zip(neg_tcrs, neg_ws)) if
               (x.split(",")[2], x.split(",")[3]) in unique_neg_cdr3_valid_set]

test_pos_f = [(x, f) for x, f in list(zip(pos_tcrs, pos_ws)) if
              (x.split(",")[2], x.split(",")[3]) in unique_pos_cdr3_test_set]
test_neg_f = [(x, f) for x, f in list(zip(neg_tcrs, neg_ws)) if
              (x.split(",")[2], x.split(",")[3]) in unique_neg_cdr3_test_set]

In [13]:
df_train_pos = pd.DataFrame(train_pos_f, columns=['tcr', 'freq'])
df_train_neg = pd.DataFrame(train_neg_f, columns=['tcr', 'freq'])

df_valid_pos = pd.DataFrame(valid_pos_f, columns=['tcr', 'freq'])
df_valid_neg = pd.DataFrame(valid_neg_f, columns=['tcr', 'freq'])

df_test_pos = pd.DataFrame(test_pos_f, columns=['tcr', 'freq'])
df_test_neg = pd.DataFrame(test_neg_f, columns=['tcr', 'freq'])

In [14]:
print("pos train proportion: "+str(df_train_pos.shape[0]/df_pos.shape[0]))
print("neg train proportion: "+str(df_train_neg.shape[0]/df_neg.shape[0]))
print("pos valid proportion: "+str(df_valid_pos.shape[0]/df_pos.shape[0]))
print("neg valid proportion: "+str(df_valid_neg.shape[0]/df_neg.shape[0]))
print("pos test proportion: "+str(df_test_pos.shape[0]/df_pos.shape[0]))
print("neg test proportion: "+str(df_test_neg.shape[0]/df_neg.shape[0]))

pos train proportion: 0.6852895148669796
neg train proportion: 0.6905200330179694
pos valid proportion: 0.1589984350547731
neg valid proportion: 0.16026414375515904
pos test proportion: 0.14710485133020346
neg test proportion: 0.14362816686773763


In [15]:
(df_train_pos.shape[0]+df_valid_pos.shape[0]+df_test_pos.shape[0])/df_pos.shape[0]

0.9913928012519562

In [16]:
(df_train_neg.shape[0]+df_valid_neg.shape[0]+df_test_neg.shape[0])/df_neg.shape[0]

0.9944123436408661

In [17]:
# write (training, validation, test)*(pos, neg) files out

df_train_pos.to_csv("./data_for_model/train_pos.csv", index=False)
df_train_neg.to_csv("./data_for_model/train_neg.csv", index=False)

df_valid_pos.to_csv("./data_for_model/valid_pos.csv", index=False)
df_valid_neg.to_csv("./data_for_model/valid_neg.csv", index=False)

df_test_pos.to_csv("./data_for_model/test_pos.csv", index=False)
df_test_neg.to_csv("./data_for_model/test_neg.csv", index=False)

In [18]:
train_pos_alpha = [x.split(",")[2] for x in df_train_pos.tcr.tolist()]
train_pos_beta = [x.split(",")[3] for x in df_train_pos.tcr.tolist()]

train_neg_alpha = [x.split(",")[2] for x in df_train_neg.tcr.tolist()]
train_neg_beta = [x.split(",")[3] for x in df_train_neg.tcr.tolist()]

valid_pos_alpha = [x.split(",")[2] for x in df_valid_pos.tcr.tolist()]
valid_pos_beta = [x.split(",")[3] for x in df_valid_pos.tcr.tolist()]

valid_neg_alpha = [x.split(",")[2] for x in df_valid_neg.tcr.tolist()]
valid_neg_beta = [x.split(",")[3] for x in df_valid_neg.tcr.tolist()]

test_pos_alpha = [x.split(",")[2] for x in df_test_pos.tcr.tolist()]
test_pos_beta = [x.split(",")[3] for x in df_test_pos.tcr.tolist()]

test_neg_alpha = [x.split(",")[2] for x in df_test_neg.tcr.tolist()]
test_neg_beta = [x.split(",")[3] for x in df_test_neg.tcr.tolist()]

In [19]:
print(set(train_pos_alpha).intersection(valid_pos_alpha))
print(set(train_pos_alpha).intersection(test_pos_alpha))
print(set(valid_pos_alpha).intersection(test_pos_alpha))

print(set(train_pos_beta).intersection(valid_pos_beta))
print(set(train_pos_beta).intersection(test_pos_beta))
print(set(valid_pos_beta).intersection(test_pos_beta))

print(set(train_neg_alpha).intersection(valid_neg_alpha))
print(set(train_neg_alpha).intersection(test_neg_alpha))
print(set(valid_neg_alpha).intersection(test_neg_alpha))

print(set(train_neg_beta).intersection(valid_neg_beta))
print(set(train_neg_beta).intersection(test_neg_beta))
print(set(valid_neg_beta).intersection(test_neg_beta))

set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
